In [ ]:
importimport os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader


In [ ]:
extract_path = r"C:\anilminiproject\datasets\WHU"

In [ ]:
class WHUDataset(Dataset):
    def __init__(self, image_dir, mask_dir):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(image_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        img_path = os.path.join(self.image_dir, self.images[index])
        mask_path = os.path.join(self.mask_dir, self.images[index])

        # Load and preprocess image
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (256, 256))
        image = image / 255.0

        # Load and preprocess mask
        mask = cv2.imread(mask_path, 0)
        mask = cv2.resize(mask, (256, 256))
        mask = mask / 255.0
        mask = np.expand_dims(mask, axis=0)

        image = torch.tensor(image, dtype=torch.float32).permute(2,0,1)
        mask = torch.tensor(mask, dtype=torch.float32)

        return image, mask



In [ ]:
# Initialize DataLoaders
train_dataset = WHUDataset(r"C:\anilminiproject\datasets\WHU\train\Image", r"C:\anilminiproject\datasets\WHU\train\Mask")
val_dataset = WHUDataset(r"C:\anilminiproject\datasets\WHU\val\Image", r"C:\anilminiproject\datasets\WHU\val\Mask")

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

In [ ]:
class DenseBlock(nn.Module):
    def __init__(self, in_channels, growth_rate=32):
        super(DenseBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, growth_rate, 3, padding=1)
        self.conv2 = nn.Conv2d(in_channels + growth_rate, growth_rate, 3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        self.out_channels = in_channels + 2 * growth_rate

    def forward(self, x):
        x1 = self.relu(self.conv1(x))
        x2 = self.relu(self.conv2(torch.cat([x, x1], dim=1)))
        return torch.cat([x, x1, x2], dim=1)

class SDSC_UNet(nn.Module):
    def __init__(self):
        super(SDSC_UNet, self).__init__()
        # Encoder
        self.enc1 = DenseBlock(3)         # 67 out
        self.pool = nn.MaxPool2d(2)
        self.enc2 = DenseBlock(67)        # 131 out
        self.enc3 = DenseBlock(131)       # 195 out
        self.bottleneck = DenseBlock(195) # 259 out

        # Decoder
        self.up3 = nn.ConvTranspose2d(259, 195, 2, stride=2)
        self.dec3 = nn.Conv2d(390, 195, 3, padding=1) # 195 + 195 skip
        self.up2 = nn.ConvTranspose2d(195, 131, 2, stride=2)
        self.dec2 = nn.Conv2d(262, 131, 3, padding=1) # 131 + 131 skip
        self.up1 = nn.ConvTranspose2d(131, 67, 2, stride=2)
        self.dec1 = nn.Conv2d(134, 64, 3, padding=1)  # 67 + 67 skip

        self.final = nn.Conv2d(64, 1, 1)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))

        d3 = self.relu(self.dec3(torch.cat([self.up3(b), e3], dim=1)))
        d2 = self.relu(self.dec2(torch.cat([self.up2(d3), e2], dim=1)))
        d1 = self.relu(self.dec1(torch.cat([self.up1(d2), e1], dim=1)))
        return self.final(d1)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SDSC_UNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss()
epochs = 25

In [ ]:
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for images, masks in tqdm(train_loader):
        images, masks = images.to(device), masks.to(device)

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, masks).item()

    print(f"\nEpoch [{epoch+1}/{epochs}] Train Loss: {train_loss/len(train_loader):.4f} Val Loss: {val_loss/len(val_loader):.4f}")

100%|██████████████████████████████████████████████████████████████████████████████| 1433/1433 [06:00<00:00,  3.97it/s]



Epoch [1/25] Train Loss: 0.1618 Val Loss: 0.1337


100%|██████████████████████████████████████████████████████████████████████████████| 1433/1433 [05:46<00:00,  4.14it/s]



Epoch [2/25] Train Loss: 0.1351 Val Loss: 0.1129


100%|██████████████████████████████████████████████████████████████████████████████| 1433/1433 [05:45<00:00,  4.15it/s]


...


Epoch [23/25] Train Loss: 0.0609 Val Loss: 0.0616


100%|██████████████████████████████████████████████████████████████████████████████| 1433/1433 [05:46<00:00,  4.14it/s]



Epoch [24/25] Train Loss: 0.0606 Val Loss: 0.0625


100%|██████████████████████████████████████████████████████████████████████████████| 1433/1433 [05:46<00:00,  4.14it/s]



Epoch [25/25] Train Loss: 0.0605 Val Loss: 0.0641


In [ ]:
torch.save(model.state_dict(),
           r"C:\anilminiproject\datasets\savedmodels\sdsc_unet__whu.pth")

print("Model saved successfully!")

Model saved successfully!
